<a href="https://colab.research.google.com/github/lynylany/algorithms/blob/main/1_dspy_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install -U dspy-ai

In [ ]:
import os
import dspy

os.environ["LITELLM_VERIFY_SSL"] = "False"
os.environ["CURL_CA_BUNDLE"] = ""

LLM_URL = 'https://llm.services.storemesh.com/v1'
MODEL_NAME = 'gpt-oss:20b'
API_KEY = 'sk-qwwRBsI8WJfS1Yh-hBgyiQ'

lm = dspy.LM(
    model=f'openai/{MODEL_NAME}',
    api_base=LLM_URL,
    api_key=API_KEY,
    cache=False
)

dspy.configure(lm=lm, track_usage=True)

print(lm('hello world!'))

InternalServerError: litellm.InternalServerError: InternalServerError: OpenAIException - Connection error.

In [ ]:
import dspy
import os
import litellm

litellm.ssl_verify = False

# URL
LLM_URL = 'https://llm.services.storemesh.com/v1'
MODEL_NAME = 'gpt-oss:20b'
API_KEY = 'sk-qwwRBsI8WJfS1Yh-hBgyiQ'

# Init LLM.
lm = dspy.LM(
    model=f'openai/{MODEL_NAME}',
    api_base=LLM_URL,
    api_key=API_KEY,
    cache=False
)
dspy.configure(lm=lm)

# 1. **Hello World**

In [ ]:
# 3. ทดสอบ
try:
    print(lm('hello world!'))
except Exception as e:
    print(f"เกิดข้อผิดพลาด: {e}")

เกิดข้อผิดพลาด: litellm.InternalServerError: InternalServerError: OpenAIException - 'Client' object has no attribute 'api_key'


# 2. **การเขียน Signature (กำหนดโครงสร้าง Input/Output)**

In [ ]:
class BasicQA(dspy.Signature):
    """ตอบคำถามที่ได้รับอย่างกระชับและตรงไปตรงมา"""
    question = dspy.InputField(desc="คำถามที่ต้องการหาคำตอบ")
    answer = dspy.OutputField(desc="คำตอบสั้นๆ 1-2 ประโยค")

### --- 1. dspy.Predict (ตอบแบบตรงไปตรงมา) ---

In [ ]:
predict_module = dspy.Predict(BasicQA)
pred_result = predict_module(question="เมืองหลวงของประเทศไทยคืออะไร?")
print(f"[Predict Output] : {pred_result.answer}\n")

### --- 2. dspy.ChainOfThought (คิดทีละสเต็ปก่อนตอบ) ---

In [ ]:
# --- 2. dspy.ChainOfThought (คิดทีละสเต็ปก่อนตอบ) ---
cot_module = dspy.ChainOfThought(BasicQA)
cot_result = cot_module(question="ถ้าฉันมีแอปเปิล 5 ผล กินไป 2 ผล แล้วซื้อเพิ่มอีก 3 ผล ฉันจะมีแอปเปิลกี่ผล?")

# แก้ไขจาก .rationale เป็น .reasoning
print(f"[CoT Reasoning] : {cot_result.reasoning}")
print(f"[CoT Output]    : {cot_result.answer}\n")

### --- 3. dspy.ReAct (คิดและลงมือทำ โดยใช้ Tools) ---

In [ ]:
def store(query: str) -> str:
    """This is store has more products!"""
    return {'coca-cola': 3, 'pepsi': 4, 'monster':1, 'ichitan': 3}

react_module = dspy.ReAct(BasicQA, tools=[store])
react_result = react_module(question="ใน store ตอนนี้มีสินค้าอะไรมั้ง ?")
print(f"[ReAct Output]  : {react_result.answer}")

In [ ]:
react_result = react_module(question="ผมต้องการโค๊ก 6 ขวด สินค้าหมดยัง")
print(f"[ReAct Output]  : {react_result.answer}")

# 3. **การทำ Teleprompt เบื้องต้น (Optimizer)**

In [ ]:
import dspy
import json

class OrderExtractor(dspy.Signature):
    """สกัดข้อมูลการสั่งซื้อจากข้อความแชทของลูกค้า
    ข้อควรระวัง: ลูกค้าอาจมีการเปลี่ยนใจ เปลี่ยนจำนวน หรือยกเลิกสินค้าบางรายการกลางทาง
    ให้สรุปเฉพาะ 'ความต้องการสุดท้าย' เท่านั้น

    ตอบกลับเป็น JSON Format เท่านั้น โครงสร้างดังนี้:
    {
      "customer_intent": "สั่งซื้อ" หรือ "สอบถาม",
      "items": [{"name": "ชื่อสินค้า", "qty": จำนวน(ตัวเลข), "note": "รายละเอียดเพิ่มเติม (เช่น สี ไซส์)"}],
      "shipping": "วิธีจัดส่ง (ถ้าไม่ระบุให้ใส่ null)"
    }"""

    chat_log = dspy.InputField(desc="ข้อความแชทจากลูกค้า")
    final_order_json = dspy.OutputField(desc="JSON String เท่านั้น ห้ามมีข้อความอื่นปน")

In [ ]:
messy_chat = """
น้องคะ พี่รับกางเกงยีนส์รุ่น J01 ไซส์ M 2 ตัวค่ะ
แล้วก็เอาหมวกบักเก็ตสีดำ 1 ใบด้วย ส่งเคอรี่ด่วนเลยนะ
เอ๊ะ เดี๋ยวๆ กางเกงยีนส์เอาแค่ตัวเดียวก่อนดีกว่า กลัวใส่ไม่ได้
ส่วนหมวกเปลี่ยนเป็นสีขาวแทนนะคะ ยอดเท่าไหร่คะเนี่ย
"""

unoptimized_module = dspy.Predict(OrderExtractor)
result = unoptimized_module(chat_log=messy_chat)
result.final_order_json

In [ ]:
import dspy
import json
from dspy.teleprompt import BootstrapFewShot

# ---------------------------------------------------------
# 1. กำหนดเป้าหมายที่ซับซ้อน (Signature)
# ---------------------------------------------------------
class OrderExtractor(dspy.Signature):
    """สกัดข้อมูลการสั่งซื้อจากข้อความแชทของลูกค้า
    ข้อควรระวัง: ลูกค้าอาจมีการเปลี่ยนใจ เปลี่ยนจำนวน หรือยกเลิกสินค้าบางรายการกลางทาง
    ให้สรุปเฉพาะ 'ความต้องการสุดท้าย' เท่านั้น

    ตอบกลับเป็น JSON Format เท่านั้น โครงสร้างดังนี้:
    {
      "customer_intent": "สั่งซื้อ" หรือ "สอบถาม",
      "items": [{"name": "ชื่อสินค้า", "qty": จำนวน(ตัวเลข), "note": "รายละเอียดเพิ่มเติม (เช่น สี ไซส์)"}],
      "shipping": "วิธีจัดส่ง (ถ้าไม่ระบุให้ใส่ null)"
    }"""

    chat_log = dspy.InputField(desc="ข้อความแชทจากลูกค้า")
    final_order_json = dspy.OutputField(desc="JSON String เท่านั้น ห้ามมีข้อความอื่นปน")

# ---------------------------------------------------------
# 2. เตรียมข้อมูลสอน (Trainset) ที่ดักทางความกำกวม
# ---------------------------------------------------------
trainset = [
    dspy.Example(
        chat_log="เอาเสื้อยืดสีดำ 2 ตัวค่ะ เอ้ยเดี๋ยว เปลี่ยนเป็นสีขาว 1 ดำ 1 ดีกว่า ส่ง EMS นะคะ",
        final_order_json='{"customer_intent": "สั่งซื้อ", "items": [{"name": "เสื้อยืด", "qty": 1, "note": "สีขาว"}, {"name": "เสื้อยืด", "qty": 1, "note": "สีดำ"}], "shipping": "EMS"}'
    ).with_inputs('chat_log'),
    dspy.Example(
        chat_log="กาแฟลาเต้เย็นแก้วนึงครับ อ้อ ไม่เอาละครับ เปลี่ยนเป็นอเมริกาโน่ร้อนแทน ไม่หวานเลยนะครับ",
        final_order_json='{"customer_intent": "สั่งซื้อ", "items": [{"name": "อเมริกาโน่ร้อน", "qty": 1, "note": "ไม่หวานเลย"}], "shipping": null}'
    ).with_inputs('chat_log')
]

# ---------------------------------------------------------
# 3. สร้าง Metric สุดโหด (ตรวจความถูกต้องของ Logic)
# ---------------------------------------------------------
def strict_order_metric(example, pred, trace=None):
    try:
        # 1. ต้องเป็น JSON ที่โหลดได้จริง
        pred_data = json.loads(pred.final_order_json)

        # 2. ต้องมีคีย์ครบถ้วน
        if not all(k in pred_data for k in ["customer_intent", "items", "shipping"]):
            return False

        # 3. เช็ก Logic ความถูกต้อง (ต้องไม่เอาของที่ลูกค้ายกเลิกมาใส่)
        # นำจำนวน qty ทั้งหมดใน items มารวมกัน แล้วเทียบกับตัวอย่าง
        expected_data = json.loads(example.final_order_json)
        pred_total_qty = sum(item.get("qty", 0) for item in pred_data.get("items", []))
        expected_total_qty = sum(item.get("qty", 0) for item in expected_data.get("items", []))

        return pred_total_qty == expected_total_qty
    except:
        return False # ถ้าพังตั้งแต่ json.loads ก็ปรับตกเลย

In [ ]:
# ---------------------------------------------------------
# 4. เริ่มกระบวนการ Optimize
# ---------------------------------------------------------
# เราใช้ ChainOfThought ให้โมเดล "คิดวิเคราะห์" ข้อความที่วกไปวนมาก่อนตัดสินใจพ่น JSON
teleprompter = BootstrapFewShot(metric=strict_order_metric, max_bootstrapped_demos=2, max_labeled_demos=1)
student = dspy.ChainOfThought(OrderExtractor)

print("กำลังเทรนโมเดลให้เข้าใจความซับซ้อน...")
optimized_extractor = teleprompter.compile(student=student, trainset=trainset)
print("เทรนเสร็จสิ้น!\n")

# ---------------------------------------------------------
# 5. ทดสอบกับโจทย์จริงที่ยากและกำกวม (Unseen Test)
# ---------------------------------------------------------
print("=== ผลลัพธ์จากการสกัดข้อมูล ===")
# ตัวแปร result จะมีทั้ง .reasoning (กระบวนการคิด) และ .final_order_json (ผลลัพธ์)
result = optimized_extractor(chat_log=messy_chat)

print("[กระบวนการคิดของ AI ก่อนตอบ]:")
print(result.reasoning)
print("\n[JSON ที่พร้อมส่งเข้า Database หรือ API]:")
print(result.final_order_json)